# Stage B2 — Train the adapter

**Experiment B — Practical application: MobileCLIP → SigLIP 2 adapter**

Goal: one linear matrix that maps iPhone-tier MobileCLIP-S1 image
embeddings into server-tier SigLIP 2 space, so a **single Qdrant
collection** serves both tiers — the phone indexes images offline, the
server queries the same index with SigLIP text embeddings.


## What this stage does
Fits two closed-form adapters (no gradient training):
- **Ridge** — a full linear map `W = (XᵀX + aI)⁻¹ XᵀY`
- **Procrustes** — a *pure rotation* from the SVD of `XᵀY`

## Why two variants
They test two strengths of the hypothesis. If Procrustes performs nearly as
well as Ridge, the spaces are identical **up to rotation** — the strongest
form of the convergence claim (rigidly same shape). Ridge alone proves the
standard form (same content up to a general linear transformation).

## Expected output
`adapter.npz`; printed held-out R² and mean cosine-to-target for both
variants. Runtime: seconds.


In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
os.environ["DATA_DIR"] = "/content/drive/MyDrive/convergence_experiment"

In [ ]:
# Configuration and imports
import numpy as np
import os
from pathlib import Path

DATA_DIR = Path(os.environ.get("DATA_DIR", "."))
DATA_DIR.mkdir(parents=True, exist_ok=True)


ALPHA = 1e-2
TRAIN_FRAC = 0.75
SEED = 0

In [ ]:
# Functions
def ridge_fit(X, Y, alpha):
    d = X.shape[1]
    W = np.linalg.solve(X.T @ X + alpha * np.eye(d), X.T @ Y)
    return W  # [d_in, d_out]


def procrustes_fit(X, Y):
    """Orthogonal map. If dims differ, lift X with zero-padding."""
    d_in, d_out = X.shape[1], Y.shape[1]
    if d_in < d_out:
        X = np.pad(X, ((0, 0), (0, d_out - d_in)))
    elif d_in > d_out:
        Y = np.pad(Y, ((0, 0), (0, d_in - d_out)))
    U, _, Vt = np.linalg.svd(X.T @ Y)
    R = U @ Vt  # [d, d] orthogonal
    return R, d_in, d_out


def apply_procrustes(X, R, d_in, d_out):
    d = R.shape[0]
    if X.shape[1] < d:
        X = np.pad(X, ((0, 0), (0, d - X.shape[1])))
    out = X @ R
    return out[:, :d_out]


def main():
    data = np.load(str(DATA_DIR / "pairs.npz"))
    X, Y = data["mob_img"], data["sig_img"]
    n = X.shape[0]

    rng = np.random.default_rng(SEED)
    idx = rng.permutation(n)
    n_tr = int(n * TRAIN_FRAC)
    tr, te = idx[:n_tr], idx[n_tr:]

    print(f"Train {len(tr)}, eval {len(te)}")
    print(f"dims: MobileCLIP {X.shape[1]} -> SigLIP {Y.shape[1]}")

    W = ridge_fit(X[tr], Y[tr], ALPHA)
    R, d_in, d_out = procrustes_fit(X[tr], Y[tr])

    def r2(Yh, Yt):
        ss_res = ((Yt - Yh) ** 2).sum()
        ss_tot = ((Yt - Yt.mean(0)) ** 2).sum()
        return 1 - ss_res / ss_tot

    def cos(Yh, Yt):
        Yh = Yh / (np.linalg.norm(Yh, axis=1, keepdims=True) + 1e-8)
        return (Yh * Yt).sum(1).mean()

    Yh_ridge = X[te] @ W
    Yh_proc = apply_procrustes(X[te], R, d_in, d_out)

    print(f"\nRidge:      held-out R2 = {r2(Yh_ridge, Y[te]):.3f}, "
          f"mean cosine to target = {cos(Yh_ridge, Y[te]):.3f}")
    print(f"Procrustes: held-out R2 = {r2(Yh_proc, Y[te]):.3f}, "
          f"mean cosine to target = {cos(Yh_proc, Y[te]):.3f}")

    np.savez_compressed(
        str(DATA_DIR / "adapter.npz"), W_ridge=W.astype(np.float32),
        R_procrustes=R.astype(np.float32),
        proc_dims=np.array([d_in, d_out]),
        train_idx=tr, eval_idx=te,
    )
    print("\nSaved adapter.npz "
          f"(ridge matrix: {W.astype(np.float32).nbytes / 1e6:.1f} MB)")

In [ ]:
# Run the training (requires pairs.npz from stage B1)
main()